In [ ]:
# Run this one cell on a Colab CPU runtime. Only file metadata is read.
from pathlib import Path
from collections import defaultdict
import json
import os


def measure_tree(root):
    root = Path(root).resolve()
    if not root.is_dir():
        raise FileNotFoundError(str(root))
    groups = defaultdict(lambda: {'files': 0, 'bytes': 0, 'videos': 0, 'video_bytes': 0})
    errors, directories = [], 0
    def failed(error):
        errors.append({'path': str(error.filename), 'error': str(error)})
    for parent, dirs, files in os.walk(root, onerror=failed, followlinks=False):
        directories += 1
        for name in list(dirs):
            p = Path(parent) / name
            if p.is_symlink():
                dirs.remove(name)
                errors.append({'path': str(p), 'error': 'symlink directory skipped'})
        for name in files:
            p = Path(parent) / name
            try:
                if p.is_symlink():
                    errors.append({'path': str(p), 'error': 'symlink file skipped'})
                    continue
                size = p.stat().st_size
                rel = p.relative_to(root)
                key = '/'.join(rel.parts[:2]) if len(rel.parts) > 2 else (rel.parts[0] if len(rel.parts) > 1 else '(root files)')
                g = groups[key]
                g['files'] += 1
                g['bytes'] += size
                if p.suffix.lower() in {'.avi', '.mp4', '.mov', '.mkv'}:
                    g['videos'] += 1
                    g['video_bytes'] += size
            except OSError as error:
                failed(error)
        if directories % 100 == 0:
            print('Đã duyệt', directories, 'thư mục,', sum(g['files'] for g in groups.values()), 'file...', flush=True)
    return dict(root=str(root), complete=not errors, directories=directories,
                total_files=sum(g['files'] for g in groups.values()),
                total_bytes=sum(g['bytes'] for g in groups.values()),
                video_files=sum(g['videos'] for g in groups.values()),
                video_bytes=sum(g['video_bytes'] for g in groups.values()),
                groups=dict(groups), errors=errors,
                note='Logical sizes of files under this folder, not Google account storage usage. No video payloads read.')


def check_drive_dataset_size():
    try:
        from google.colab import drive
    except ImportError:
        print('Chọn runtime Colab CPU; kernel này là local nên chưa truy cập được Drive của bạn.')
        return
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    # Change only this path if the dataset is stored elsewhere.
    DATASET_DIR = '/content/drive/MyDrive/Datasets/IKEA_ASM'
    print('Đếm metadata của', DATASET_DIR, '— không tải video, không cần GPU.', flush=True)
    result = measure_tree(DATASET_DIR)
    print('HOÀN TẤT' if result['complete'] else 'CHƯA ĐỦ: có file/thư mục chưa đọc được')
    print('Tổng:', result['total_files'], 'file |', f"{result['total_bytes']/1e9:.3f} GB / {result['total_bytes']/2**30:.3f} GiB")
    print('Video:', result['video_files'], 'file |', f"{result['video_bytes']/1e9:.3f} GB")
    for group, v in sorted(result['groups'].items()):
        print(f"{group}: {v['bytes']/1e9:.3f} GB | {v['files']} file | {v['videos']} video")
    print('Lỗi/bỏ qua:', len(result['errors']))
    if result['errors']:
        print(json.dumps(result['errors'][:10], ensure_ascii=False, indent=2))
    report = Path('/content/ikea_drive_size_report.json')
    report.write_text(json.dumps(result, ensure_ascii=False, indent=2))
    print('Báo cáo local:', report)


check_drive_dataset_size()
